# Integration tests
Test how HTTP, validation, and storage work together.


In [ ]:
from fastapi import FastAPI
from httpx import ASGITransport, AsyncClient

app = FastAPI()

@app.post("/tasks", status_code=201)
async def create_task() -> dict[str, int]:
    return {"id": 1}

async with AsyncClient(transport=ASGITransport(app=app), base_url="http://test") as client:
    response = await client.post("/tasks")
assert response.status_code == 201
print(response.json())


## Polished version
Build the real application around an in-memory adapter and exercise it through HTTP.


In [ ]:
from typing import Protocol
from pydantic import BaseModel, ConfigDict

class TaskCreate(BaseModel):
    title: str

class TaskView(TaskCreate):
    model_config = ConfigDict(from_attributes=True)
    id: int

class TaskRepository(Protocol):
    async def add(self, title: str) -> TaskView: ...

class MemoryTaskRepository:
    def __init__(self) -> None:
        self.tasks: list[TaskView] = []
    async def add(self, title: str) -> TaskView:
        task = TaskView(id=len(self.tasks) + 1, title=title)
        self.tasks.append(task)
        return task

def create_service(repository: TaskRepository) -> FastAPI:
    service = FastAPI()
    @service.post("/tasks", response_model=TaskView, status_code=201)
    async def create_task(payload: TaskCreate) -> TaskView:
        return await repository.add(payload.title)
    return service

repository = MemoryTaskRepository()
service = create_service(repository)
async with AsyncClient(transport=ASGITransport(app=service), base_url="http://test") as client:
    response = await client.post("/tasks", json={"title": "Ship API"})
assert response.status_code == 201
assert response.json() == {"title": "Ship API", "id": 1}
assert len(repository.tasks) == 1
print(response.json())
